**生成式 AI 使用说明**：本作业中使用生成式 AI 工具时，适用与协作相同的课程政策。和其他协作者一样，每位学生都必须独立写出自己的解答，不能直接依赖交互输出；提交内容中还应注明协作的性质。使用生成式 AI 工具实质性完成作业内容不符合本作业的精神，也会违反 [Honor Code](https://communitystandards.stanford.edu/policies-and-guidance/honor-code)。

In [ ]:
# 将你的 Google Drive 挂载到 Colab VM。
from google.colab import drive
drive.mount('/content/drive')

# TODO：填写 Drive 中保存解压后
# 作业文件夹，例如 'cs231n/assignments/assignment1/'
FOLDERNAME = 'cs231n/assignments/assignment1/'
assert FOLDERNAME is not None, "[!] Enter the foldername."

# 现在已经挂载 Drive，下面确保
# Colab VM 的 Python 解释器可以加载
# 其中的 Python 文件。
import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

# 将 CIFAR-10 数据集下载到你的 Drive
# 如果它还不存在。
%cd /content/drive/My\ Drive/$FOLDERNAME/cs231n/datasets/
!bash get_datasets.sh
%cd /content/drive/My\ Drive/$FOLDERNAME

# 全连接神经网络
在这个练习中，我们将用模块化方式实现全连接网络。对每一层，我们都会实现一个 `forward` 函数和一个 `backward` 函数。`forward` 函数接收输入、权重和其他参数，并返回输出以及一个 `cache` 对象；`cache` 保存反向传播所需的数据，例如：

```python
def layer_forward(x, w):
  """接收输入 x 和权重 w"""
  # 做一些计算 ...
  z = # ... 某个中间值
  # 再做一些计算 ...
  out = # 输出

  cache = (x, w, z, out) # 计算梯度时需要的值

  return out, cache
```

反向传播会接收上游导数和 `cache` 对象，并返回关于输入和权重的梯度，例如：

```python
def layer_backward(dout, cache):
  """
  接收 dout（损失对输出的导数）和 cache，
  并计算对输入的导数。
  """
  # 解包 cache 中的值
  x, w, z, out = cache

  # 使用 cache 中的值计算导数
  dx = # 损失对 x 的导数
  dw = # 损失对 w 的导数

  return dx, dw
```

用这种方式实现多个层之后，我们就能很容易地组合它们，构建不同架构的分类器。

In [ ]:
# 和往常一样，先做一些初始化设置。
from __future__ import print_function
import time
import numpy as np
import matplotlib.pyplot as plt
from cs231n.classifiers.fc_net import *
from cs231n.data_utils import get_CIFAR10_data
from cs231n.gradient_check import eval_numerical_gradient, eval_numerical_gradient_array
from cs231n.solver import Solver

%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0) # 设置图像的默认大小
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

# 用于自动重新加载外部模块
# 参考 http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
%load_ext autoreload
%autoreload 2

def rel_error(x, y):
  """ returns relative error """
  return np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))

In [ ]:
# 加载预处理后的 CIFAR-10 数据。

data = get_CIFAR10_data()
for k, v in list(data.items()):
  print(('%s: ' % k, v.shape))

# Affine 层：前向传播
打开 `cs231n/layers.py` 文件，实现 `affine_forward` 函数。

完成后，可以运行下面的代码测试你的实现：

In [ ]:
# 测试 affine_forward 函数

num_inputs = 2
input_shape = (4, 5, 6)
output_dim = 3

input_size = num_inputs * np.prod(input_shape)
weight_size = output_dim * np.prod(input_shape)

x = np.linspace(-0.1, 0.5, num=input_size).reshape(num_inputs, *input_shape)
w = np.linspace(-0.2, 0.3, num=weight_size).reshape(np.prod(input_shape), output_dim)
b = np.linspace(-0.3, 0.1, num=output_dim)

out, _ = affine_forward(x, w, b)
correct_out = np.array([[ 1.49834967,  1.70660132,  1.91485297],
                        [ 3.25553199,  3.5141327,   3.77273342]])

# 将你的输出与参考输出比较，误差应约为 e-9 或更小。
print('Testing affine_forward function:')
print('difference: ', rel_error(out, correct_out))

# Affine 层：反向传播
现在实现 `affine_backward` 函数，并使用数值梯度检查来测试你的实现。

In [ ]:
# 测试 affine_backward 函数
np.random.seed(231)
x = np.random.randn(10, 2, 3)
w = np.random.randn(6, 5)
b = np.random.randn(5)
dout = np.random.randn(10, 5)

dx_num = eval_numerical_gradient_array(lambda x: affine_forward(x, w, b)[0], x, dout)
dw_num = eval_numerical_gradient_array(lambda w: affine_forward(x, w, b)[0], w, dout)
db_num = eval_numerical_gradient_array(lambda b: affine_forward(x, w, b)[0], b, dout)

_, cache = affine_forward(x, w, b)
dx, dw, db = affine_backward(dout, cache)

# 误差应约为 e-10 或更小。
print('Testing affine_backward function:')
print('dx error: ', rel_error(dx_num, dx))
print('dw error: ', rel_error(dw_num, dw))
print('db error: ', rel_error(db_num, db))

# ReLU 激活：前向传播
在 `relu_forward` 函数中实现 ReLU 激活函数的前向传播，并使用下面的代码测试你的实现：

In [ ]:
# 测试 relu_forward 函数

x = np.linspace(-0.5, 0.5, num=12).reshape(3, 4)

out, _ = relu_forward(x)
correct_out = np.array([[ 0.,          0.,          0.,          0.,        ],
                        [ 0.,          0.,          0.04545455,  0.13636364,],
                        [ 0.22727273,  0.31818182,  0.40909091,  0.5,       ]])

# 将你的输出与参考输出比较，误差应在 e-8 量级。
print('Testing relu_forward function:')
print('difference: ', rel_error(out, correct_out))

# ReLU 激活：反向传播
现在在 `relu_backward` 函数中实现 ReLU 激活函数的反向传播，并使用数值梯度检查测试你的实现：

In [ ]:
np.random.seed(231)
x = np.random.randn(10, 10)
dout = np.random.randn(*x.shape)

dx_num = eval_numerical_gradient_array(lambda x: relu_forward(x)[0], x, dout)

_, cache = relu_forward(x)
dx = relu_backward(dout, cache)

# 误差应在 e-12 量级。
print('Testing relu_backward function:')
print('dx error: ', rel_error(dx_num, dx))

## 内联问题 1：

我们只要求你实现 ReLU，但神经网络中还有许多不同的激活函数，每种都有优缺点。激活函数中常见的一个问题是在反向传播时得到零梯度（或接近零的梯度）流。下列哪些激活函数会有这个问题？如果考虑一维情形，什么类型的输入会导致这种行为？
1. Sigmoid
2. ReLU
3. Leaky ReLU

$\color{blue}{\textit 你的回答：}$ *在此填写*

# “三明治”层
神经网络中经常会出现一些常用的层组合模式。例如，affine 层后面通常接一个 ReLU 非线性。为了方便使用这些常见模式，我们在 `cs231n/layer_utils.py` 文件中定义了几个便利层。

现在先查看 `affine_relu_forward` 和 `affine_relu_backward` 函数，然后运行下面的代码，对反向传播做数值梯度检查：

In [ ]:
from cs231n.layer_utils import affine_relu_forward, affine_relu_backward
np.random.seed(231)
x = np.random.randn(2, 3, 4)
w = np.random.randn(12, 10)
b = np.random.randn(10)
dout = np.random.randn(2, 10)

out, cache = affine_relu_forward(x, w, b)
dx, dw, db = affine_relu_backward(dout, cache)

dx_num = eval_numerical_gradient_array(lambda x: affine_relu_forward(x, w, b)[0], x, dout)
dw_num = eval_numerical_gradient_array(lambda w: affine_relu_forward(x, w, b)[0], w, dout)
db_num = eval_numerical_gradient_array(lambda b: affine_relu_forward(x, w, b)[0], b, dout)

# 相对误差应约为 e-10 或更小。
print('Testing affine_relu_forward and affine_relu_backward:')
print('dx error: ', rel_error(dx_num, dx))
print('dw error: ', rel_error(dw_num, dw))
print('db error: ', rel_error(db_num, db))

# 损失层：Softmax
现在在 `cs231n/layers.py` 的 `softmax_loss` 函数中实现 softmax 的损失和梯度。这应该与你在 `cs231n/classifiers/softmax.py` 中实现的内容类似。其他损失函数（例如 `svm_loss`）也可以用模块化方式实现，但本作业不要求。

运行下面的代码可以确认实现是否正确：

In [ ]:
np.random.seed(231)
num_classes, num_inputs = 10, 50
x = 0.001 * np.random.randn(num_inputs, num_classes)
y = np.random.randint(num_classes, size=num_inputs)


dx_num = eval_numerical_gradient(lambda x: softmax_loss(x, y)[0], x, verbose=False)
loss, dx = softmax_loss(x, y)

# 测试 softmax_loss 函数。损失应接近 2.3，dx 误差应约为 e-8。
print('\nTesting softmax_loss:')
print('loss: ', loss)
print('dx error: ', rel_error(dx_num, dx))

# 两层网络
打开 `cs231n/classifiers/fc_net.py` 文件，完成 `TwoLayerNet` 类的实现。通读代码，确保你理解它的 API。你可以运行下面的单元来测试实现。

In [ ]:
np.random.seed(231)
N, D, H, C = 3, 5, 50, 7
X = np.random.randn(N, D)
y = np.random.randint(C, size=N)

std = 1e-3
model = TwoLayerNet(input_dim=D, hidden_dim=H, num_classes=C, weight_scale=std)

print('Testing initialization ... ')
W1_std = abs(model.params['W1'].std() - std)
b1 = model.params['b1']
W2_std = abs(model.params['W2'].std() - std)
b2 = model.params['b2']
assert W1_std < std / 10, 'First layer weights do not seem right'
assert np.all(b1 == 0), 'First layer biases do not seem right'
assert W2_std < std / 10, 'Second layer weights do not seem right'
assert np.all(b2 == 0), 'Second layer biases do not seem right'

print('Testing test-time forward pass ... ')
model.params['W1'] = np.linspace(-0.7, 0.3, num=D*H).reshape(D, H)
model.params['b1'] = np.linspace(-0.1, 0.9, num=H)
model.params['W2'] = np.linspace(-0.3, 0.4, num=H*C).reshape(H, C)
model.params['b2'] = np.linspace(-0.9, 0.1, num=C)
X = np.linspace(-5.5, 4.5, num=N*D).reshape(D, N).T
scores = model.loss(X)
correct_scores = np.asarray(
  [[11.53165108,  12.2917344,   13.05181771,  13.81190102,  14.57198434, 15.33206765,  16.09215096],
   [12.05769098,  12.74614105,  13.43459113,  14.1230412,   14.81149128, 15.49994135,  16.18839143],
   [12.58373087,  13.20054771,  13.81736455,  14.43418138,  15.05099822, 15.66781506,  16.2846319 ]])
scores_diff = np.abs(scores - correct_scores).sum()
assert scores_diff < 1e-6, 'Problem with test-time forward pass'

print('Testing training loss (no regularization)')
y = np.asarray([0, 5, 1])
loss, grads = model.loss(X, y)
correct_loss = 3.4702243556
assert abs(loss - correct_loss) < 1e-10, 'Problem with training-time loss'

model.reg = 1.0
loss, grads = model.loss(X, y)
correct_loss = 26.5948426952
assert abs(loss - correct_loss) < 1e-10, 'Problem with regularization loss'

# 相对误差应约为 e-7 或更小。
for reg in [0.0, 0.7]:
  print('Running numeric gradient check with reg = ', reg)
  model.reg = reg
  loss, grads = model.loss(X, y)

  for name in sorted(grads):
    f = lambda _: model.loss(X, y)[0]
    grad_num = eval_numerical_gradient(f, model.params[name], verbose=False)
    print('%s relative error: %.2e' % (name, rel_error(grad_num, grads[name])))

# Solver
打开 `cs231n/solver.py` 文件并阅读代码，熟悉它的 API。之后，使用一个 `Solver` 实例训练 `TwoLayerNet`，使其在验证集上达到约 `36%` 的准确率。

In [ ]:
input_size = 32 * 32 * 3
hidden_size = 50
num_classes = 10
model = TwoLayerNet(input_size, hidden_size, num_classes)
solver = None

##############################################################################
# TODO：使用 Solver 实例训练一个 TwoLayerNet，使其在验证集上达到约 36% 准确率。 #
# 在验证集上的准确率。                                            #
##############################################################################

##############################################################################
#                             你的代码结束                               #
##############################################################################

# 调试训练过程
使用上面提供的默认参数时，你应该在验证集上得到约 0.36 的准确率。这并不算好。

理解问题的一种策略是在优化过程中绘制损失函数、训练集准确率和验证集准确率。

另一种策略是可视化网络第一层学到的权重。大多数在视觉数据上训练的神经网络，其第一层权重在可视化时通常会呈现某些可见结构。

In [ ]:
# 运行这个单元，可视化训练损失以及训练/验证准确率。

plt.subplot(2, 1, 1)
plt.title('Training loss')
plt.plot(solver.loss_history, 'o')
plt.xlabel('Iteration')

plt.subplot(2, 1, 2)
plt.title('Accuracy')
plt.plot(solver.train_acc_history, '-o', label='train')
plt.plot(solver.val_acc_history, '-o', label='val')
plt.plot([0.5] * len(solver.val_acc_history), 'k--')
plt.xlabel('Epoch')
plt.legend(loc='lower right')
plt.gcf().set_size_inches(15, 12)
plt.show()

In [ ]:
from cs231n.vis_utils import visualize_grid

# 可视化 权重 网络的

def show_net_weights(net):
    W1 = net.params['W1']
    W1 = W1.reshape(3, 32, 32, -1).transpose(3, 1, 2, 0)
    plt.imshow(visualize_grid(W1, padding=3).astype('uint8'))
    plt.gca().axis('off')
    plt.show()

show_net_weights(model)

# 调优超参数

**哪里不对？** 从上面的可视化可以看到，损失大致线性下降，这似乎表明学习率可能太低。此外，训练准确率和验证准确率之间没有差距，说明我们使用的模型容量较低，应该增大模型规模。另一方面，如果模型非常大，我们预期会看到更多过拟合，表现为训练准确率和验证准确率之间有很大的差距。

**调参。** 调整超参数并培养对它们如何影响最终性能的直觉，是使用神经网络的重要部分，所以我们希望你多加练习。下面你应该尝试不同的超参数取值，包括隐藏层大小、学习率、训练 epoch 数量和正则化强度。你也可以考虑调学习率衰减，不过使用默认值也应该能得到不错的性能。

**近似结果。** 你的目标是在验证集上达到高于 48% 的分类准确率。我们的最佳网络在验证集上超过 52%。

**实验：** 本练习的目标是用一个全连接神经网络在 CIFAR-10 上尽可能取得好结果（52% 可以作为参考）。你可以自由实现自己的技术，例如用 PCA 降维、添加 dropout、给 solver 增加功能等。

In [ ]:
best_model = None



#################################################################################
# TODO：使用验证集调优超参数。将你训练得到的最佳模型存入 best_net。       #
# 存入 best_net。                                                          #
#                                                                               #
# 为了帮助调试网络，你可以使用类似上面那些可视化结果。                 #
# 对于调得较差的网络，这些可视化结果会出现明显的定性差异。             #
#
#                                                                               #
# 手动调超参数也可以，但你可能会发现写代码自动遍历超参数组合更有用，   #
# 就像前面练习中做过的那样。                                         #
#
#################################################################################

################################################################################
#                              你的代码结束                                #
################################################################################

# 测试你的模型
在验证集和测试集上运行你的最佳模型。你应该在验证集和测试集上都达到 48% 以上的准确率。

In [ ]:
y_val_pred = np.argmax(best_model.loss(data['X_val']), axis=1)
print('Validation set accuracy: ', (y_val_pred == data['y_val']).mean())

In [ ]:
y_test_pred = np.argmax(best_model.loss(data['X_test']), axis=1)
print('Test set accuracy: ', (y_test_pred == data['y_test']).mean())

In [ ]:
# 保存最佳模型
best_model.save("best_two_layer_net.npy")

## 内联问题 2：

现在你已经训练了一个神经网络分类器，可能会发现测试准确率远低于训练准确率。我们可以通过哪些方式缩小这个差距？请选择所有适用项。

1. 在更大的数据集上训练。
2. 添加更多隐藏单元。
3. 增大正则化强度。
4. 以上都不是。

$\color{blue}{\textit 你的回答：}$

$\color{blue}{\textit 你的解释：}$